In [1]:
import os
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image

In [2]:
#connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
MODEL_PATH = "/content/drive/MyDrive/AML/final_xgboost_model.pkl"
PREPROCESS_PATH = "/content/drive/MyDrive/AML/preprocessing.pkl"

xgb_model = joblib.load(MODEL_PATH)
preprocess_objects = joblib.load(PREPROCESS_PATH)

scaler_img = preprocess_objects["scaler_img"]
pca = preprocess_objects["pca"]
scaler_clinical = preprocess_objects["scaler_clinical"]
clinical_feature_names = preprocess_objects["clinical_feature_names"]
label_map = preprocess_objects["label_map"]

In [4]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


In [5]:
def extract_single_image_feature(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    feat = base_model.predict(img_array, verbose=0)
    return feat.flatten()

In [6]:
def extract_patient_image_feature(patient_image_paths):
    all_features = []

    for img_path in patient_image_paths:
        feat = extract_single_image_feature(img_path)
        all_features.append(feat)

    if len(all_features) == 0:
        raise ValueError("No valid patient images were found.")

    all_features = np.array(all_features)
    patient_feature = np.mean(all_features, axis=0).reshape(1, -1)

    return patient_feature

In [7]:
def predict_new_patient(patient_image_paths, clinical_data_dict):
    missing_cols = [col for col in clinical_feature_names if col not in clinical_data_dict]
    if missing_cols:
        raise ValueError(f"Missing clinical features: {missing_cols}")

    # IMAGE BRANCH
    img_features = extract_patient_image_feature(patient_image_paths)
    img_scaled = scaler_img.transform(img_features)
    img_pca = pca.transform(img_scaled)

    # CLINICAL BRANCH
    clinical_df = pd.DataFrame([clinical_data_dict])
    clinical_df = clinical_df[clinical_feature_names]
    clinical_values = clinical_df.values
    clinical_scaled = scaler_clinical.transform(clinical_values)

    # COMBINE
    X_new_final = np.concatenate([img_pca, clinical_scaled], axis=1)

    # PREDICT
    pred_class = int(xgb_model.predict(X_new_final)[0])
    pred_prob = xgb_model.predict_proba(X_new_final)[0]
    pred_label = label_map[pred_class]

    return pred_class, pred_label, pred_prob

In [8]:
patient_folder = "/content/drive/MyDrive/AML/Images/control/AEC"
MAX_IMAGES = 5

patient_image_paths = []

for file in os.listdir(patient_folder):
    if file.lower().endswith(".tif"):
        patient_image_paths.append(os.path.join(patient_folder, file))

patient_image_paths = patient_image_paths[:MAX_IMAGES]

print("Loaded images:", len(patient_image_paths))
print(patient_image_paths)

Loaded images: 5
['/content/drive/MyDrive/AML/Images/control/AEC/image_0.tif', '/content/drive/MyDrive/AML/Images/control/AEC/image_100.tif', '/content/drive/MyDrive/AML/Images/control/AEC/image_1.tif', '/content/drive/MyDrive/AML/Images/control/AEC/image_10.tif', '/content/drive/MyDrive/AML/Images/control/AEC/image_101.tif']


In [9]:
new_clinical_data = {
    "sex_1f_2m": 1,
    "age": 25,
    "leucocytes_per_ul": 6.2,
    "pb_myeloblast": 0,
    "pb_promyelocyte": 0,
    "pb_myelocyte": 0,
    "pb_metamyelocyte": 0,
    "pb_neutrophil_band": 3,
    "pb_neutrophil_segmented": 55,
    "pb_eosinophil": 3,
    "pb_basophil": 1,
    "pb_monocyte": 6,
    "pb_lymph_typ": 30,
    "pb_lymph_atyp_react": 2,
    "pb_other": 0
}

In [10]:
pred_class, pred_label, pred_prob = predict_new_patient(
    patient_image_paths,
    new_clinical_data
)

print("Predicted class:", pred_class)
print("Predicted label:", pred_label)
print("Probabilities:", pred_prob)

Predicted class: 0
Predicted label: Control
Probabilities: [0.9171375  0.04988192 0.01545144 0.01158241 0.00594671]


In [11]:
label_names = ["Control", "NPM1", "PML_RARA", "RUNX1_RUNX1T1", "CBFB_MYH11"]
prob_dict = {label_names[i]: float(pred_prob[i]) for i in range(len(label_names))}

print("Predicted label:", pred_label)
print("Probabilities:", prob_dict)

Predicted label: Control
Probabilities: {'Control': 0.9171375036239624, 'NPM1': 0.04988192394375801, 'PML_RARA': 0.015451439656317234, 'RUNX1_RUNX1T1': 0.011582409031689167, 'CBFB_MYH11': 0.0059467097744345665}
